In [1]:
# Creating the Spark session
from pyspark.sql import SparkSession
import spark

spark = SparkSession.builder \
        .appName("OlistData") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 13:11:49 INFO SparkEnv: Registering MapOutputTracker
26/04/22 13:11:50 INFO SparkEnv: Registering BlockManagerMaster
26/04/22 13:11:50 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/22 13:11:50 INFO SparkEnv: Registering OutputCommitCoordinator


### Reading all the data files

In [2]:
# to simplfy I am writing thhe hdfs_path and calling the files in that folder
hdfs_path = '/tmp/brazilian-ecommerce/'

In [3]:
## Reading the customer data file
## With inferSchema as true, Spark automatically detects and assigns appropriate column data types based on the data

customers_df= spark.read.format("csv").option("header","true").option("inferSchema", "true").load(hdfs_path+'olist_customers_dataset.csv')

## Reading the olist_orders_dataset file
orders_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_orders_dataset.csv')

## Reading the olist_order_items_dataset file
order_item_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_items_dataset.csv')

## Reading the order_payments file
payments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_payments_dataset.csv')

## Reading the olist_order_reviews_dataset file
reviews_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_reviews_dataset.csv')

## Reading the olist_products_dataset file
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)

## Reading the olist_sellers_dataset file
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)

## Reading the olist_geolocation_dataset file
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)

## Reading the olist_orders_dataset file
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)

In [4]:
# Caching Frequently used Data for Better Performance

customers_df.cache()
orders_df.cache()
order_item_df.cache()

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [5]:
### Joining the orders_df and order_item_df data

orders_item_joined_df = orders_df.join(order_item_df, 'order_id', 'inner')

In [6]:
### Joining the orders_item_joined_df with products_df data

orders_item_products_df = orders_item_joined_df.join(products_df, 'product_id', 'inner')

In [7]:
### Joining the orders_item_products_df with sellers_df data

orders_items_products_sellers_df = orders_item_products_df.join(sellers_df,'seller_id','inner')

In [8]:
### Joining the orders_items_products_sellers_df with customers_df data

full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

In [9]:
# Geolocation Data # Don't want to miss the data that doesn't exists in geolocation data

full_orders_df = full_orders_df.join(geolocation_df, full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [10]:
# I have to keep all the data; if reviews_df doesnot have some data then it's going to miss so doing left

full_orders_df = full_orders_df.join(reviews_df, 'order_id', 'left')

In [11]:
# I have to keep all the data; if payments_df doesnot have some data then it's going to miss so doing left

full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [12]:
## Doing the full_orders_df cache 
full_orders_df.cache()

26/04/22 13:14:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [13]:
## Find the total Revenue Per Seller
from pyspark.sql.functions import *

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price').alias('total_revenue'))
seller_revenue_df.show(6)

+--------------------+--------------------+
|           seller_id|       total_revenue|
+--------------------+--------------------+
|7a67c85e85bb2ce85...|2.0312794890000038E7|
|9d213f303afae4983...|   2321.400000000004|
|d2374cbcbb3ca4ab1...|   3375517.550000011|
|1835b56ce799e6a4d...|   6097995.110000008|
|d650b663c3b5f6fb3...|           2253869.1|
|aded58c8142dedc54...|    62478.0000000005|
+--------------------+--------------------+
only showing top 6 rows



## Optimized Joins For Data integration

1. A broadcast join is used to optimize join performance when one of the tables is small.
2. In a regular join: Spark performs a shuffle. Data moves across the cluster. This is slow and expensive

In [14]:
### Joining the orders_df and order_item_df data
orders_item_joined_df = orders_df.join(order_item_df, 'order_id', 'inner')

### Joining the orders_item_joined_df with products_df data
orders_item_products_df = orders_item_joined_df.join(products_df, 'product_id', 'inner')


In [15]:
# so for the small data tables we are doing the broadcast
orders_items_products_sellers_df = orders_item_products_df.join(broadcast(sellers_df),'seller_id','inner')

In [16]:
### Joining the orders_items_products_sellers_df with customers_df data
full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

# GEolocation Data
full_orders_df = full_orders_df.join(broadcast(geolocation_df),full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [17]:
full_orders_df = full_orders_df.join(broadcast(reviews_df),'order_id','left')

# I have to keep all the data; if payments_df doesnot have some data then it's going to miss so doing left
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [18]:
full_orders_df.cache()

26/04/22 13:16:13 WARN CacheManager: Asked to cache already cached data.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

## Aggregation

In [19]:
# Total number of Orders Per Customer

customer_order_count_df = full_orders_df.groupBy('customer_id').agg(count('order_id').alias('total_orders')).orderBy(desc('total_orders'))
customer_order_count_df.show(6)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        6876|
|d3e82ccec3cb5f956...|        6876|
+--------------------+------------+
only showing top 6 rows



In [20]:
# Average Review Score Per Seller

seller_review_df = full_orders_df.groupBy('seller_id').agg(avg('review_score').alias('avg_score')).orderBy(desc('avg_score'))
seller_review_df.show(6)

+--------------------+---------+
|           seller_id|avg_score|
+--------------------+---------+
|0b36063d5818f81cc...|      5.0|
|404e1ba01358af4cd...|      5.0|
|01ed254b9ff8407df...|      5.0|
|0ade5cc4a305ed709...|      5.0|
|28c7d8743fbc8679f...|      5.0|
|d3891911c2feae53c...|      5.0|
+--------------------+---------+
only showing top 6 rows



In [21]:
# Most Sold Products ( Top 10 )

top_sold_products = full_orders_df.groupBy('product_id').agg(count('order_id').alias('sold_product'))
top_sold_products.orderBy(col('sold_product').desc()).limit(10).show()

+--------------------+------------+
|          product_id|sold_product|
+--------------------+------------+
|aca2eb7d00ea1a7b8...|       86740|
|422879e10f4668299...|       81110|
|99a4788cb24856965...|       78775|
|389d119b48cf3043d...|       60248|
|d1c427060a0f73f6b...|       59274|
|368c6c730842d7801...|       58358|
|53759a2ecddad2bb8...|       52654|
|53b36df67ebb7c415...|       52105|
|154e7e31ebfa09220...|       42700|
|3dd2a17168ec895c7...|       40787|
+--------------------+------------+



In [22]:
## Top 10 Customer By Spending

top_spend_customer = full_orders_df.groupBy('customer_id').agg(sum('price').alias('total_spend'))
top_spend_customer.orderBy(col('total_spend').desc()).limit(10).show()

+--------------------+------------------+
|         customer_id|       total_spend|
+--------------------+------------------+
|d3e82ccec3cb5f956...|         6662844.0|
|df55c14d1476a9a34...|         3565657.0|
|fe5113a38e3575c04...|         3293604.0|
|ec5b2ba62e5743423...|         2556120.0|
|63b964e79dee32a35...|         2501664.0|
|46bb3c0b1a65c8399...|         2336752.0|
|05455dfa7cd02f13d...| 2160194.400000087|
|3690e975641f01bd0...|         2124498.0|
|349509b216bd5ec11...|         1923627.0|
|695476b5848d64ba0...|1820543.1299999943|
+--------------------+------------------+



## Window Function and Ranking

In [23]:
from pyspark.sql.window import Window


# Dense Rank for Sellers Based on Revenue

window_spec = Window.partitionBy('seller_id').orderBy(desc('price'))

In [25]:
# Rank Top Selling Products Per seller

top_seller_products_df = full_orders_df.withColumn('rank',rank().over(window_spec)).filter(col('rank')<=5)

top_seller_products_df.select('seller_id','price','rank').show()

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
+--------------------+-----+----+
only showing top 20 rows



## Advance Aggregation and Enrichment

In [26]:
full_orders_df

DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [27]:
# Total Revenue & Average Order Value (AOV) per Customer

customer_spending_df = full_orders_df.groupBy('customer_id') \
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_spent'),
        round(avg('price'),2).alias('AOV') 
    ).orderBy(desc('total_spent'))

customer_spending_df.show()

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spent|    AOV|
+--------------------+------------+------------------+-------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|
|695476b5848d64ba0...|         687|1820543.1299999943|2649.99|
|73236a0796f53d60d...|         832|         1755520.0| 2110.0|
|cc803a2c412833101...|         762|         1676400.0| 2200.0|
|1ff773612ab8934db...|        5820|1658641.7999999512| 

In [28]:
# test for AOV
6662844/6876

969.0

In [29]:
# Seller Performance Metrics ( Revenue, Average Review, Order Count)

seller_performance_df = full_orders_df.groupBy('seller_id') \
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue'),
        round(avg('review_score'),2).alias('avg_review_score'),
        round(stddev('price'),2).alias('price_variability')
    ).orderBy(desc('total_revenue'))


seller_performance_df.show()

+--------------------+------------+--------------------+----------------+-----------------+
|           seller_id|total_orders|       total_revenue|avg_review_score|price_variability|
+--------------------+------------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|      184587| 3.613871731999995E7|            4.09|           111.65|
|53243585a1d6dc264...|       54514| 3.429159294999999E7|            4.12|           499.65|
|4a3ca9315b744ce9f...|      330661|  3.37595708400001E7|            3.77|            59.37|
|7c67e1448b00f6e96...|      233306|3.2282321789999764E7|            3.42|            50.39|
|fa1c13f2614d7b5c4...|       87686|3.0139386309999973E7|            4.38|            307.7|
|da8622b14eb17ae28...|      264433|2.9857669730000045E7|            3.98|            72.92|
|7e93a43ef30c4f03f...|       50226|2.6315706299999956E7|            4.15|           377.24|
|1025f0e2d44d7041d...|      229587|2.2937518519999973E7|            3.89|       

In [30]:
# Product Popularity Metrics

product_metrics_df = full_orders_df.groupBy('product_id') \
    .agg(
        count('order_id').alias('total_sales'),
        sum('price').alias('total_revenue'),
        round(avg('price'),2).alias('avg_price'),
        round(stddev('price'),2).alias('price_volatility'),\
        collect_set('seller_id').alias('unique_sellers')    #collect_set returns a list of distinct values for each group
    ).orderBy(desc('total_sales'))

product_metrics_df.show()

+--------------------+-----------+------------------+---------+----------------+--------------------+
|          product_id|total_sales|     total_revenue|avg_price|price_volatility|      unique_sellers|
+--------------------+-----------+------------------+---------+----------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740| 6164630.300000013|    71.07|            3.17|[955fee9216a65b61...|
|422879e10f4668299...|      81110|  4442791.51000001|    54.77|            4.46|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775| 6921762.710000018|    87.87|            4.08|[4a3ca9315b744ce9...|
|389d119b48cf3043d...|      60248| 3280533.130000013|    54.45|            4.37|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274| 8220103.329999989|   138.68|           16.58|[a1043bafd471dff5...|
|368c6c730842d7801...|      58358| 3181698.899999992|    54.52|            4.59|[1f50f920176fa81d...|
|53759a2ecddad2bb8...|      52654| 2893017.499999999|    54.94|            4.52|[1

In [31]:
## Monthly Revenue and order Count

revenue_order_metrics_df = full_orders_df.withColumn("order_month", date_format(col("order_purchase_timestamp"), "MM")) \
        .groupBy("order_month") \
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue'),
        round(avg('price'),2).alias('avg_order_value'),
        round(min('price'),2).alias('min_order_value'),
        round(max('price'),2).alias('max_order_value')
    ).orderBy(desc('total_revenue'))

revenue_order_metrics_df.show()

+-----------+------------+--------------------+---------------+---------------+---------------+
|order_month|total_orders|       total_revenue|avg_order_value|min_order_value|max_order_value|
+-----------+------------+--------------------+---------------+---------------+---------------+
|         05|     1918571|2.4006115196999753E8|         125.12|            3.5|         6499.0|
|         08|     1914313|2.3195870784999895E8|         121.17|            2.2|        4399.87|
|         07|     1847639|2.2290885709999844E8|         120.65|            1.2|         6729.0|
|         03|     1809467| 2.186811684300005E8|         120.85|            4.9|        4099.99|
|         04|     1693860|2.1715696912999958E8|          128.2|           0.85|         4799.0|
|         06|     1701909|2.1024332348999992E8|         123.53|           3.49|         4590.0|
|         02|     1551163| 1.787817840699995E8|         115.26|           2.99|         6735.0|
|         01|     1495580|1.715329014999

In [32]:
# Customer Retention Analysis ( First & Last Order )

customer_retention_df = full_orders_df.groupBy('customer_id')\
    .agg(
        first('order_purchase_timestamp').alias('first_order_date'),
        last('order_purchase_timestamp').alias('last_order_date'),
        count('order_id').alias('total_orders'),
        round(avg('price'),2).alias('aov')
    ).orderBy(desc('total_orders'))

customer_retention_df.show()

+--------------------+-------------------+-------------------+------------+------+
|         customer_id|   first_order_date|    last_order_date|total_orders|   aov|
+--------------------+-------------------+-------------------+------------+------+
|351e40989da90e704...|2017-07-13 10:42:37|2017-07-13 10:42:37|       11427| 85.99|
|50920f8cd0681fd86...|2018-01-27 11:28:32|2018-01-27 11:28:32|       10752| 43.82|
|9b43e2a62de9bab3a...|2017-05-25 22:27:50|2017-05-25 22:27:50|        8556|  26.4|
|270c23a11d024a44c...|2017-08-08 20:26:31|2017-08-08 20:26:31|        8001| 36.59|
|5c87184371002d49e...|2018-01-05 19:15:37|2018-01-05 19:15:37|        6876| 12.49|
|d3e82ccec3cb5f956...|2017-03-18 14:28:34|2017-03-18 14:28:34|        6876| 969.0|
|d5f2b3f597c7ccafb...|2017-12-13 14:21:15|2017-12-13 14:21:15|        6706|  59.0|
|c2f18647725395af4...|2018-03-06 19:21:47|2018-03-06 19:21:47|        6612|  34.9|
|24e7dc2ff8c071263...|2017-11-24 16:16:45|2017-11-24 16:16:45|        6597|  59.2|
|7bb

## Extended Enrichment 

In [33]:
## Order Status Flag

full_orders_df = full_orders_df.withColumn('is_delivered', when(col('order_status')== 'delivered',lit(1)).otherwise(lit(0))) \
                               .withColumn('is_canceled', when(col('order_status')== 'canceled',lit(1)).otherwise(lit(0)))

full_orders_df.where(full_orders_df['order_status']=='canceled').select('order_status','is_delivered','is_canceled').show()

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
+------------+------------+-----------+
only showing top 20 rows



In [34]:
# Order Revenue Calcualtion

full_orders_df = full_orders_df.withColumn('order_revenue', col('price')+col('freight_value'))

full_orders_df.select('price', 'freight_value', 'order_revenue').show()

+-----+-------------+------------------+
|price|freight_value|     order_revenue|
+-----+-------------+------------------+
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
|28.99|         7.46|36.449999999999996|
+-----+-------------+------------------+
only showing top

In [35]:
customer_spending_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_spent: double (nullable = true)
 |-- AOV: double (nullable = true)



In [36]:
# Customer Segmentation based on spending

customer_spending_df = customer_spending_df.withColumn('customer_segment', 
                                                       when(col('AOV') >= 1200, 'High-Value') \
                                                       .when( (col('AOV') >= 500) & (col('AOV') < 1200), 'Medium-Value') \
                                                       .otherwise('Low-Value'))

customer_spending_df.show()

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|    Medium-Value|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|      High-Value|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|      High-Value|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|      High-Value|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|       Low-Value|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|      High-Value|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|    Medium-Value|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|      High-Value|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|      High-Value|
|695476b5848d64ba0...|         687|18205

In [37]:
full_orders_df = full_orders_df.join(customer_spending_df, 'customer_id', 'left')

full_orders_df.select('customer_id','customer_segment').show()

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
|88b0bac2d79ffc975...|       Low-Value|
+--------------------+----------------+
only showing top 20 rows



In [38]:

full_orders_df.select('order_purchase_timestamp').show(2)

+------------------------+
|order_purchase_timestamp|
+------------------------+
|     2018-08-07 23:19:16|
|     2018-08-07 23:19:16|
+------------------------+
only showing top 2 rows



In [39]:
## Hourly Order Distribution # getting the hour from order_purchase_timestamp

full_orders_df = full_orders_df.withColumn('hour_of_day', expr('hour(order_purchase_timestamp)'))

full_orders_df.select('order_purchase_timestamp','hour_of_day').show()

+------------------------+-----------+
|order_purchase_timestamp|hour_of_day|
+------------------------+-----------+
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
|     2018-08-07 23:19:16|         23|
+------------------------+-----------+
only showing top 20 rows



In [40]:
# Weekday vs Weekend Order

full_orders_df = full_orders_df.withColumn('order_day_type', when(dayofweek('order_purchase_timestamp').isin(1,7),lit('Weekend')).otherwise(lit('weekday')))

full_orders_df.select('order_purchase_timestamp', 'order_day_type').show()

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
|     2018-08-07 23:19:16|       weekday|
+------------------------+--------

In [41]:
## Frieght Category
# full_orders_df.select('freight_value').distinct().show() # =< 10 Low, >10 and <20 Medium, >=20 High


## Frieght Category
full_orders_df = full_orders_df.withColumn('frieght_category', when(col('freight_value') >= 20, 'High-Value') \
                                                               .when( (col('freight_value') > 10) & (col('freight_value') < 20), 'Medium-Value') \
                                                               .otherwise('Low-Value'))

full_orders_df.select('freight_value', 'frieght_category').distinct().show()

+-------------+----------------+
|freight_value|frieght_category|
+-------------+----------------+
|         5.73|       Low-Value|
|        20.69|      High-Value|
|        18.23|    Medium-Value|
|        14.52|    Medium-Value|
|        18.65|    Medium-Value|
|       107.02|      High-Value|
|        19.35|    Medium-Value|
|        17.74|    Medium-Value|
|        11.18|    Medium-Value|
|        24.11|      High-Value|
|        16.11|    Medium-Value|
|        15.07|    Medium-Value|
|        13.71|    Medium-Value|
|        18.42|    Medium-Value|
|        28.82|      High-Value|
|         19.5|    Medium-Value|
|        21.35|      High-Value|
|        18.16|    Medium-Value|
|        18.38|    Medium-Value|
|        16.14|    Medium-Value|
+-------------+----------------+
only showing top 20 rows



In [42]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

### Saving in to the hadoop folder

In [43]:
!hadoop fs -mkdir /tmp/olist

mkdir: `/tmp/olist': File exists


In [44]:
full_orders_df.write.mode('overwrite').parquet('/tmp/olist')

In [47]:
!hadoop fs -ls -h /tmp/olist

Found 11 items
-rw-r--r--   2 root hadoop          0 2026-04-22 13:28 /tmp/olist/_SUCCESS
-rw-r--r--   2 root hadoop      8.8 M 2026-04-22 13:27 /tmp/olist/part-00000-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      8.1 M 2026-04-22 13:27 /tmp/olist/part-00001-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      7.6 M 2026-04-22 13:27 /tmp/olist/part-00002-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      9.4 M 2026-04-22 13:27 /tmp/olist/part-00003-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      9.3 M 2026-04-22 13:28 /tmp/olist/part-00004-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      8.4 M 2026-04-22 13:28 /tmp/olist/part-00005-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop      9.7 M 2026-04-22 13:28 /tmp/olist/part-00006-dabc22f8-8568-4bc1-bfe0-c22b3